# Przetworzenie obrazów przez GroundingSAM

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Dodanie dźwięku przy długich komórkach
import subprocess
def play_sound():
    sound_file = '/mnt/d/Backup/INZ/msg.ogg'
    subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

In [6]:
sys.path.append(str(Path().resolve().parent.parent / "Grounded-SAM-2"))

from torchvision.ops import box_convert
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from grounding_dino.groundingdino.util.inference import load_model, load_image, predict
import torch

2026-03-17 17:47:06.145334: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-17 17:47:07.692607: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Ustawienie opcji wyświetlania w pandas (opcjonalne)

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [49]:
SAVE_DIR = Path("/mnt/d/Backup/MAGISTERSKIE/outputs/GDSAM_172")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Pobranie danych

Pobranie ścieżek do plików w odpowiedniej kolejności. Następnie połączenie z danymi w pliku .csv tak, aby sobie odpowidały.

In [51]:
DATA_PATH = Path("/home/ruszczka/projekty/test_files/img")

# Uniwersalna funkcja do przetwarzania każdego podfolderu
def prepare_image_dataframe(folder_name: str, file_prefix: str) -> pd.DataFrame:
    # Poprawne użycie pathlib (łączenie ścieżek operatorem '/')
    folder_path = DATA_PATH / folder_name
    csv_path = folder_path / f"{file_prefix}.csv"
    
    # Wczytanie id z csv
    df = pd.read_csv(csv_path, sep=" ", usecols=["id"]).rename(columns={"id": "image_id"})
    
    # Dodanie źródła
    df.insert(1, "source", folder_name)
    
    # Zamiast wolnego .apply(), używamy list comprehension.
    # Jest to znacznie szybsze i bardziej "pythoniczne" w tym przypadku.
    df['color_path'] = [folder_path / f"{file_prefix}_{img_id}_color_orig.png" for img_id in df['image_id']]
    df['depth_path'] = [folder_path / f"{file_prefix}_{img_id}_depth.png" for img_id in df['image_id']]
    
    return df

# Parametry naszych dwóch zbiorów
configs = [
    {"folder": "icm_111", "prefix": "imitation_control_model_2025_03_25-14_39_50"},
    {"folder": "icm_61",  "prefix": "imitation_control_model_2024_12_03-12_31_55"}
]

# Przetwarzamy wszystkie foldery w pętli i od razu łączymy je w jeden DataFrame
dataframes = [prepare_image_dataframe(cfg["folder"], cfg["prefix"]) for cfg in configs]
df = pd.concat(dataframes, ignore_index=True)

print(f"Ilość danych: {len(df)}")
df.head()

Ilość danych: 172


,image_id,source,color_path,depth_path
0,0,icm_111,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_0_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_0_depth.png
1,1,icm_111,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_1_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_1_depth.png
2,2,icm_111,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_2_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_2_depth.png
3,3,icm_111,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_3_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_3_depth.png
4,4,icm_111,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_4_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_111/imitation_control_model_2025_03_25-14_39_50_4_depth.png


In [52]:
df.tail()

,image_id,source,color_path,depth_path
167,56,icm_61,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_56_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_56_depth.png
168,57,icm_61,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_57_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_57_depth.png
169,58,icm_61,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_58_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_58_depth.png
170,59,icm_61,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_59_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_59_depth.png
171,60,icm_61,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_60_color_orig.png,/home/ruszczka/projekty/test_files/img/icm_61/imitation_control_model_2024_12_03-12_31_55_60_depth.png


Zapisanie dataframe do pliku .csv

In [ ]:
# df.to_csv(SAVE_DIR / 'df.csv', index=False)

# Konfiguracja GDSAM

In [54]:
df = pd.read_csv(SAVE_DIR / 'df.csv')
df.shape

(172, 4)

In [46]:
# Wczytanie do stałych ścieżek do modeli i konfiguracji
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM2_CHECKPOINT = "Grounded-SAM-2/checkpoints/sam2.1_hiera_large.pt"
GROUNDING_DINO_CONFIG = "Grounded-SAM-2/grounding_dino/groundingdino/config/GroundingDINO_SwinB_cfg.py"
GROUNDING_DINO_CHECKPOINT = "Grounded-SAM-2/gdino_checkpoints/groundingdino_swinb_cogcoor.pth"

# Ustawienie odpowiednich parametrów
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.get_device_properties(0).major >= 8:
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [55]:
# załadaowanie modelu SAM
sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
sam2_predictor = SAM2ImagePredictor(sam2_model)

# załadaowanie modelu GD
grounding_model = load_model(
    model_config_path=GROUNDING_DINO_CONFIG, 
    model_checkpoint_path=GROUNDING_DINO_CHECKPOINT,
    device=DEVICE
)

final text_encoder_type: bert-base-uncased


# Detekcja i segmentacja

In [ ]:
incomplete_detect = []
masks_array = []
test = 20
all_boxes = np.zeros((test,3,4), dtype=float) if test < len(df) else np.zeros((len(df),3,4), dtype=float)

# Tekstowe prompty dla detekcji
TEXT_PROMPTS = ["pink robot finger gripper on the robot gripper.", "green small rectangle on the robot gripper.", "blue robot finger gripper on the robot gripper."]

# Próg detekcji dla boxów i tekstu
BOX_TRESHOLD = 0.28
TEXT_TRESHOLD = 0.30

# Parametr odfiltrowania dużych masek
AREA_THRESHOLD = 2300

for count_iter, i in enumerate(range(0, 20)):

  # Załadowanie obrazu kolorowego
  image_source, image = load_image(df['color_path'][i])
  h, w, _ = image_source.shape
  sam2_predictor.set_image(image_source)

  # Załadowanie obrazu głębokości
  depth_img, _ = load_image(df['depth_path'][i])
  depth_img = depth_img[:, :, 0]

  # Tymczasowa tablica do przechowywania masek
  tmp = np.zeros((len(TEXT_PROMPTS), h, w), dtype=np.float32)

  # Iteracja po trzech obiektach
  for j in range(len(TEXT_PROMPTS)):

    # Wybór tekstu dla obiektu
    text_prompt = TEXT_PROMPTS[j]

    # Detekcja obiektów
    boxes, confidences, labels = predict(
        model=grounding_model,
        image=image,
        caption=text_prompt,
        box_threshold=BOX_TRESHOLD,
        text_threshold=TEXT_TRESHOLD)

    # Konwersja boxów z cxcywh na xyxy
    boxes2 = boxes * torch.Tensor([w, h, w, h])
    xyxy = box_convert(boxes=boxes2, in_fmt="cxcywh", out_fmt="xyxy").numpy()

    # Sprawdzenie czy są detekcje
    print(f"Obraz {i}, Obiekt {j}: Liczba wykrytych boxów: {len(xyxy)}")
    print(f"xyxy = {xyxy}")
    if xyxy.size == 0:
        print(f"Brak detekcji dla '{text_prompt}' na obrazie {i}")
        incomplete_detect.append(i)
        continue

    # Filtrowanie boxów po rozmiarze (przed segmentacją)
    max_wh = 80 if j == 1 else 60
    valid_box_indices = []
    for idx in range(len(xyxy)):
        box_width = xyxy[idx][2] - xyxy[idx][0]
        box_height = xyxy[idx][3] - xyxy[idx][1]
        if box_width <= max_wh and box_height <= max_wh:
            valid_box_indices.append(idx)

    # Sprawdzenie czy pozostały jakieś boxy po filtrowaniu
    if not valid_box_indices:
      print(f"Obraz {i}, Obiekt {j}: Wszystkie boxy przekraczają limit {max_wh}px")
      incomplete_detect.append(i)
      continue

    # Wybór tylko przefiltrowanych boxów do segmentacji
    valid_box_indices = np.array(valid_box_indices)
    filtered_xyxy = xyxy[valid_box_indices]
    filtered_confidences = confidences[valid_box_indices]
    filtered_labels = [labels[idx] for idx in valid_box_indices]

    print(f"Po filtrowaniu pozostało {len(filtered_xyxy)} boxów")

    # Segmentacja obiektów (tylko na przefiltrowanych boxach)
    masks, scores, logits = sam2_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=filtered_xyxy,
        multimask_output=False,
    )

    """
    Pzetwarzanie po detekcji i segmentacji
    """

    # Zamiana wymiarów do (n, H, W)
    if masks.ndim == 4:
        masks = masks.squeeze(1)

    # Obliczenie pola segmentacji
    mask_areas = np.sum(masks, axis=(1, 2))  # Obliczenie pola segmentacji
    valid_mask_indices = np.where(mask_areas < AREA_THRESHOLD)[0] # Zachowanie segmentacji poniżej 2300 pikseli

    # Sprawdzenie czy są maski
    if len(valid_mask_indices) == 0:
        print(f"Obraz {i}, Obiekt {j}: Brak maski z polem < {AREA_THRESHOLD} pikseli")
        incomplete_detect.append(i)
        continue 

    # Wybór maski o największym współczynniku pewności (z pozostałych kandydatów)
    filtered_confidences_np = filtered_confidences.numpy()
    valid_confidences = filtered_confidences_np[valid_mask_indices]
    best_mask_idx = valid_mask_indices[np.argmax(valid_confidences)]

    # Wybór najlepszej maski i boxa
    selected_mask = masks[best_mask_idx:best_mask_idx+1]  # Zachowanie wymiarów (1, H, W)
    selected_xyxy = filtered_xyxy[best_mask_idx:best_mask_idx+1]  # Zachowanie wymiarów (1, 4)
    selected_confidence = filtered_confidences_np[best_mask_idx]
    selected_label = filtered_labels[best_mask_idx]
    selected_area = mask_areas[best_mask_idx]

    box_width = selected_xyxy[0][2] - selected_xyxy[0][0]
    box_height = selected_xyxy[0][3] - selected_xyxy[0][1]

    # Zapisanie boxa odpowiadającego wybranej masce (znormalizowane wartości)
    normalized_box = selected_xyxy[0].copy()
    normalized_box[0] /= w  # x_min
    normalized_box[1] /= h  # y_min
    normalized_box[2] /= w  # x_max
    normalized_box[3] /= h  # y_max
    all_boxes[i][j] = normalized_box

    # Wyświetlenie informacji
    print(f"Numer obrazu: {i}")
    print(f"Tekst wejściowy: {text_prompt}")
    print(f"Maska - Pole: {selected_area} pikseli, Pewność: {selected_confidence:.2f}")
    print(f"Box - Szerokość: {box_width:.1f}px, Wysokość: {box_height:.1f}px")
    print(f"Box znormalizowany: {normalized_box}")

    # Zapisanie maski do tablicy
    mask = selected_mask[0]
    tmp[j] = mask

    """
    Wizualizacja i wyświetlanie
    """

        # # Zmienne do wizualizacji
        # class_ids = np.array([0]) 
        # labels = [f"{selected_label} {selected_confidence:.2f}"]

        # # Detekcje do wizuazlizacji
        # detections = sv.Detections(
        #     xyxy=selected_xyxy,  # (1, 4)
        #     mask=selected_mask.astype(bool),  # (1, H, W)
        #     class_id=class_ids
        # )

        # # Wizualizacja prostokątów
        # box_annotator = sv.BoxAnnotator()
        # annotated_frame = box_annotator.annotate(scene=cv2.cvtColor(image_source, cv2.COLOR_BGR2RGB), detections=detections)

        # # Wizualizacja etykiet
        # label_annotator = sv.LabelAnnotator()
        # annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

        # # Wizualizacja masek
        # mask_annotator = sv.MaskAnnotator()
        # annotated_frame = mask_annotator.annotate(scene=annotated_frame, detections=detections)
        # sv.plot_image(annotated_frame)

  # Zapisanie masek do tablicy
  masks_array.append(tmp)

masks_array = np.array(masks_array)
print(f"masks_array -> {masks_array.shape}")

print(f"Lista niekompletnych id: {incomplete_detect}")

# Niekompletne detekcjeprint(f"Liczba niekompletnych id: {incomplete_detect.size}. Liczba iteracji: {count_iter+1}. Procent odrzuconych: {incomplete_detect.size/(count_iter+1)*100}%")

incomplete_detect = np.array(list(set(incomplete_detect)))
incomplete_detect = np.unique(incomplete_detect)

print(f"Liczba niekompletnych id: {incomplete_detect.size}. Liczba iteracji: {count_iter+1}. Procent odrzuconych: {incomplete_detect.size/(count_iter+1)*100}%")
print(f"Lista niekompletnych id: {incomplete_detect}")

play_sound()

Obraz 0, Obiekt 0: Liczba wykrytych boxów: 3
xyxy = [[286.7528   115.42283  304.34058  132.3482  ]
 [279.27148  113.48024  312.8872   140.51772 ]
 [  0.590683 135.08127  510.72723  381.16898 ]]
Po filtrowaniu pozostało 2 boxów
Numer obrazu: 0
Tekst wejściowy: pink robot finger gripper on the robot gripper.
Maska - Pole: 458.0 pikseli, Pewność: 0.38
Box - Szerokość: 33.6px, Wysokość: 27.0px
Box znormalizowany: [0.5454521  0.26764208 0.6111078  0.3314097 ]
Obraz 0, Obiekt 1: Liczba wykrytych boxów: 1
xyxy = [[  0.516037 134.80237  510.43835  380.42426 ]]
Obraz 0, Obiekt 1: Wszystkie boxy przekraczają limit 80px
Obraz 0, Obiekt 2: Liczba wykrytych boxów: 1
xyxy = [[279.24176 114.62065 315.1311  141.20401]]
Po filtrowaniu pozostało 1 boxów
Numer obrazu: 0
Tekst wejściowy: blue robot finger gripper on the robot gripper.
Maska - Pole: 461.0 pikseli, Pewność: 0.31
Box - Szerokość: 35.9px, Wysokość: 26.6px
Box znormalizowany: [0.54539406 0.2703317  0.61549044 0.33302832]
Obraz 1, Obiekt 0: Lic

Zapisanie tablic z danymi do pliku

In [ ]:
# np.save(SAVE_DIR / 'masks_array.npy', masks_array)
# np.save(SAVE_DIR / 'incomplete_detect.npy', incomplete_detect)
# np.save(SAVE_DIR / 'all_boxes.npy', all_boxes)

In [201]:
print(f"Ilość niekompletnych danych: {len(incomplete_detect)}\nProcent niekompletnych danych: {len(incomplete_detect)/len(all_boxes)*100:.2f}%")

Ilość niekompletnych danych: 58
Procent niekompletnych danych: 33.72%


# wizualizacja masek i boxów

In [200]:
masks_array = np.load(SAVE_DIR / 'masks_array.npy')
incomplete_detect = np.load(SAVE_DIR / 'incomplete_detect_manual.npy', allow_pickle=True).tolist()
all_boxes = np.load(SAVE_DIR / 'all_boxes.npy')

Manualne sprawdzenie, które były poprawne

In [ ]:
# ID do usunięcia po sprawdzniu ręcznym: 12, 20, 30, 43, 79, 80, 94, 95, 100, 102, 39, 46

# incomplete_detect = set(incomplete_detect)
# for i in [12, 20, 30, 43, 79, 80, 94, 95, 100, 102, 39, 46]: incomplete_detect.add(i)
# complete_detect = set(range(0,len(all_boxes))).difference(set(incomplete_detect))
# incomplete_detect = np.array(incomplete_detect)
# np.save(SAVE_DIR / 'incomplete_detect_manual.npy' ,incomplete_detect)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import random #random.randint(0,len(incomplete_detect)-1)

for i in incomplete_detect:
  TEST_IDX = i#random.randint(0,all_boxes.shape[0]-1)
  test_row = df.iloc[TEST_IDX]

  # Wczytanie obrazu kolorowego
  color_image = cv2.imread(str(test_row['color_path']))
  color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
  h, w = color_image.shape[:2]

  # Tworzenie obrazu z nałożonymi maskami
  plt.figure(figsize=(15, 10))
  plt.imshow(color_image)

  # Nałożenie masek z przezroczystością - z wygenerowanych danych
  red_mask = masks_array[TEST_IDX][0]  # Pink gripper
  green_mask = masks_array[TEST_IDX][1]  # Green rectangle
  blue_mask = masks_array[TEST_IDX][2]  # Blue gripper

  plt.imshow(red_mask, cmap='Reds', alpha=0.35)
  plt.imshow(green_mask, cmap='Greens', alpha=0.35)
  plt.imshow(blue_mask, cmap='Blues', alpha=0.35)

  # Rysowanie bounding boxów z wygenerowanych danych
  colors = ['red', 'green', 'blue']
  labels_text = ['Czerwony', 'Zielony', 'Niebieski']

  for i, color in enumerate(colors):
    x_min_norm, y_min_norm, x_max_norm, y_max_norm = all_boxes[TEST_IDX][i]
    
    # Sprawdzenie czy box jest niepusty
    if x_min_norm == 0 and y_min_norm == 0 and x_max_norm == 0 and y_max_norm == 0:
      print(f"Box {i} jest pusty")
      continue
    
    # Konwersja ze znormalizowanych do pikseli
    x_min_pixel = int(x_min_norm * w)
    y_min_pixel = int(y_min_norm * h)
    x_max_pixel = int(x_max_norm * w)
    y_max_pixel = int(y_max_norm * h)
    
    rect = patches.Rectangle((x_min_pixel, y_min_pixel),
                            x_max_pixel - x_min_pixel,
                            y_max_pixel - y_min_pixel,
                            linewidth=2, edgecolor=color, facecolor='none',
                            label=f"{labels_text[i]} element")
    plt.gca().add_patch(rect)

  plt.title(f"Obraz o ID: {test_row['image_id']} - Maski i otaczające prostokąty")
  plt.legend(loc='upper right')
  plt.axis('off')
  plt.tight_layout()
  plt.show()